# 02 · SIH/SUS — Demanda Hospitalar (KPI 1)

Cada linha é uma AIH: uma internação paga pelo SUS. Aqui saem volume, sazonalidade,
permanência, mortalidade e — o que sustenta o IPA — o **fluxo intermunicipal**.

In [1]:
from pathlib import Path
import pandas as pd, pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
pd.set_option('display.max_columns', 60, 'display.width', 200)

def ler(sistema, arquivo, colunas=None):
    """Le um parquet de data/processed. `colunas` evita carregar as 113/208 colunas inteiras."""
    return pd.read_parquet(PROC / sistema / f'{arquivo}.parquet', columns=colunas)

def ler_varios(sistema, glob='*', colunas=None):
    arqs = sorted((PROC / sistema).glob(f'{glob}.parquet'))
    return pd.concat([pd.read_parquet(a, columns=colunas).assign(_arquivo=a.stem) for a in arqs],
                     ignore_index=True)

sorted(p.name for p in PROC.iterdir())

['CNES', 'SIHSUS', 'SIM', 'SINASC']

In [2]:
COLS = ['MUNIC_RES','MUNIC_MOV','CNES','N_AIH','DT_INTER','DT_SAIDA','DIAS_PERM',
        'DIAG_PRINC','PROC_REA','ESPEC','CAR_INT','COMPLEX','MORTE','IDADE','COD_IDADE',
        'SEXO','UTI_MES_TO','VAL_TOT']
sih = ler_varios('SIHSUS', 'RDSP*', COLS)
sih['DT_INTER'] = pd.to_datetime(sih.DT_INTER, format='%Y%m%d', errors='coerce')
sih['DT_SAIDA'] = pd.to_datetime(sih.DT_SAIDA, format='%Y%m%d', errors='coerce')
for c in ['DIAS_PERM','MORTE','UTI_MES_TO','VAL_TOT','IDADE']:
    sih[c] = pd.to_numeric(sih[c], errors='coerce')
sih['competencia'] = sih.DT_INTER.dt.to_period('M')
print(f'{len(sih):,} internações · {sih.competencia.min()} a {sih.competencia.max()}')
sih.head(3)

5,860,558 internações · 2008-01 a 2026-05


,MUNIC_RES,MUNIC_MOV,CNES,N_AIH,DT_INTER,DT_SAIDA,DIAS_PERM,DIAG_PRINC,PROC_REA,ESPEC,CAR_INT,COMPLEX,MORTE,IDADE,COD_IDADE,SEXO,UTI_MES_TO,VAL_TOT,_arquivo,competencia
0,355030,355030,2077523,3524114417340,2024-05-01,2024-05-03,2,C679,0416010172,01,02,03,0,54,4,1,0,1040.42,RDSP2406,2024-05
1,355030,355030,2077523,3524114427911,2024-06-09,2024-06-22,13,O998,0303100044,02,02,02,0,29,4,3,0,341.72,RDSP2406,2024-06
2,355030,355030,2077523,3524114427922,2024-06-20,2024-06-23,3,O829,0411010034,02,02,02,0,25,4,3,0,672.11,RDSP2406,2024-06


## Volume e sazonalidade

In [3]:
mensal = sih.groupby('competencia').agg(
    internacoes=('N_AIH','size'), obitos=('MORTE','sum'),
    perm_media=('DIAS_PERM','mean'), valor=('VAL_TOT','sum'))
mensal['letalidade_%'] = 100 * mensal.obitos / mensal.internacoes
display(mensal.round(2))
mensal.internacoes.plot(figsize=(13,3), marker='o', title='Internações por mês — SP');

,internacoes,obitos,perm_media,valor,letalidade_%
competencia,,,,,
2008-01,15488,53,30.33,3.321259e+07,0.34
2008-02,157,1,30.25,3.303093e+05,0.64
2008-03,72,0,30.42,1.546359e+05,0.00
2008-04,56,1,30.20,1.194015e+05,1.79
2008-05,72,0,30.42,1.546359e+05,0.00
...,...,...,...,...,...
2026-01,237616,11503,4.83,4.543889e+08,4.84
2026-02,224018,10486,4.72,4.170756e+08,4.68
2026-03,255793,11311,4.41,4.746791e+08,4.42


ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

## Fluxo intermunicipal

`MUNIC_RES` (onde mora) ≠ `MUNIC_MOV` (onde internou). A diferença mede dependência:
município que exporta muito paciente não tem oferta local — sinal direto de pressão.

In [ ]:
sih['fora'] = sih.MUNIC_RES != sih.MUNIC_MOV
print(f'internações fora do município de residência: {100*sih.fora.mean():.1f}%')

fluxo = (sih.groupby('MUNIC_RES')
            .agg(internacoes=('N_AIH','size'), fora=('fora','sum'))
            .assign(evasao_pct=lambda d: 100*d.fora/d.internacoes)
            .query('internacoes >= 500')
            .sort_values('evasao_pct', ascending=False))
fluxo.head(15).round(1)

In [ ]:
# principais pares origem -> destino
(sih[sih.fora].groupby(['MUNIC_RES','MUNIC_MOV']).size()
     .sort_values(ascending=False).head(15).rename('internacoes').reset_index())

## Perfil clínico

In [ ]:
top = sih.DIAG_PRINC.value_counts().head(20).rename('internacoes').to_frame()
top['%'] = (100*top.internacoes/len(sih)).round(2)
top['capitulo_cid'] = top.index.str[0]
top

In [ ]:
# letalidade e permanência por diagnóstico frequente
(sih.groupby('DIAG_PRINC')
    .agg(n=('N_AIH','size'), letalidade=('MORTE','mean'), perm=('DIAS_PERM','mean'))
    .query('n >= 2000')
    .assign(letalidade=lambda d: (100*d.letalidade).round(1), perm=lambda d: d.perm.round(1))
    .sort_values('letalidade', ascending=False).head(15))

## Hospitais

In [ ]:
(sih.groupby('CNES')
    .agg(internacoes=('N_AIH','size'), perm=('DIAS_PERM','mean'),
         letalidade=('MORTE','mean'), uti=('UTI_MES_TO', lambda s: (s>0).mean()))
    .query('internacoes >= 1000')
    .assign(perm=lambda d: d.perm.round(1), letalidade=lambda d: (100*d.letalidade).round(1),
            uti_pct=lambda d: (100*d.uti).round(1)).drop(columns='uti')
    .sort_values('internacoes', ascending=False).head(15))

## Qualidade dos dados

O que precisa virar regra de tratamento na camada Silver.

In [ ]:
print('nulos por coluna (%):')
display((100*sih[COLS].isna().mean()).round(2).sort_values(ascending=False).head(10))
print('\nDIAS_PERM negativo:', (sih.DIAS_PERM < 0).sum())
print('DT_SAIDA < DT_INTER:', (sih.DT_SAIDA < sih.DT_INTER).sum())
print('IDADE > 120 (COD_IDADE=4=anos):', ((sih.COD_IDADE=='4') & (sih.IDADE > 120)).sum())
print('MUNIC_RES fora de SP (não 35xxxx):', (~sih.MUNIC_RES.astype(str).str.startswith('35')).sum(),
      '— legítimo: residente de outro estado internado em SP')